# Working with Two SQL Server Instances

This notebook demonstrates how to move data between two SQL Server instances:
- **Primary Instance** (localhost:1433) - Source data
- **Secondary Instance** (localhost:1434) - Target data

**Common scenarios:**
- Testing data migration workflows
- Simulating source → staging → production pipelines
- Data replication testing
- ETL process development

## Step 1: Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, current_timestamp
import os

# Create Spark session
spark = SparkSession.builder \
    .appName("DualSQLServer") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark Session Created!")
print(f"Spark Version: {spark.version}")

## Step 2: Configure Connections to Both SQL Servers

In [ ]:
# Primary SQL Server configuration
primary_jdbc_url = "jdbc:sqlserver://sqlserver:1433;databaseName=FabricDev;encrypt=false;trustServerCertificate=true"
primary_properties = {
    "user": "sa",
    "password": "YourStrong@Passw0rd",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Secondary SQL Server configuration
secondary_jdbc_url = "jdbc:sqlserver://sqlserver2:1433;databaseName=master;encrypt=false;trustServerCertificate=true"
secondary_properties = {
    "user": "sa",
    "password": "YourStrong@Passw0rd",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("✅ Both SQL Server connections configured")
print(f"Primary:   {primary_jdbc_url}")
print(f"Secondary: {secondary_jdbc_url}")

## Step 3: Create Database on Secondary Instance

In [ ]:
# Note: We can't execute CREATE DATABASE via JDBC directly
# Run this in terminal or SQL client:
# docker exec fabric-sqlserver2 /opt/mssql-tools/bin/sqlcmd -S localhost -U sa -P "YourStrong@Passw0rd" -Q "CREATE DATABASE FabricDev2"

print("ℹ️  To create database on secondary instance, run this command:")
print('docker exec fabric-sqlserver2 /opt/mssql-tools/bin/sqlcmd -S localhost -U sa -P "YourStrong@Passw0rd" -Q "CREATE DATABASE FabricDev2"')
print("\nOr connect with Azure Data Studio to localhost,1434 and create manually")

## Step 4: Read Data from Primary SQL Server

In [ ]:
# Read Sales table from primary instance
df_sales_primary = spark.read.jdbc(
    url=primary_jdbc_url,
    table="dbo.Sales",
    properties=primary_properties
)

print(f"✅ Data loaded from PRIMARY SQL Server")
print(f"Total rows: {df_sales_primary.count()}")
print(f"\nSample data:")
df_sales_primary.show(5, truncate=False)

## Step 5: Transform Data

Add a timestamp and create aggregated version

In [ ]:
from pyspark.sql.functions import lit, current_timestamp

# Add migration timestamp
df_transformed = df_sales_primary.withColumn(
    "MigratedAt", 
    current_timestamp()
).withColumn(
    "SourceServer",
    lit("Primary")
)

print("✅ Data transformed with metadata")
df_transformed.select("SaleID", "CustomerID", "TotalAmount", "MigratedAt", "SourceServer").show(5)

## Step 6: Write Data to Secondary SQL Server

**Note:** You need to create FabricDev2 database first (see Step 3)

In [ ]:
# Update JDBC URL to use FabricDev2 database
secondary_jdbc_url_dev2 = "jdbc:sqlserver://sqlserver2:1433;databaseName=FabricDev2;encrypt=false;trustServerCertificate=true"

# Write to secondary SQL Server
try:
    df_transformed.write.jdbc(
        url=secondary_jdbc_url_dev2,
        table="dbo.Sales",
        mode="overwrite",
        properties=secondary_properties
    )
    print("✅ Data written to SECONDARY SQL Server!")
    print("Table: FabricDev2.dbo.Sales")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure FabricDev2 database exists on secondary instance!")
    print("Run: docker exec fabric-sqlserver2 /opt/mssql-tools/bin/sqlcmd -S localhost -U sa -P 'YourStrong@Passw0rd' -Q 'CREATE DATABASE FabricDev2'")

## Step 7: Verify Data on Secondary Instance

In [ ]:
# Read back from secondary to verify
try:
    df_verify = spark.read.jdbc(
        url=secondary_jdbc_url_dev2,
        table="dbo.Sales",
        properties=secondary_properties
    )
    
    print("✅ Data verified on SECONDARY SQL Server")
    print(f"Total rows: {df_verify.count()}")
    print("\nSample with migration metadata:")
    df_verify.select("SaleID", "ProductName", "TotalAmount", "MigratedAt", "SourceServer").show(5, truncate=False)
except Exception as e:
    print(f"❌ Could not read from secondary: {e}")

## Step 8: Compare Data Between Instances

In [ ]:
# Count comparison
primary_count = df_sales_primary.count()

try:
    secondary_count = df_verify.count()
    
    print("📊 Data Comparison:")
    print(f"Primary instance:   {primary_count} rows")
    print(f"Secondary instance: {secondary_count} rows")
    print(f"Match: {'✅ Yes' if primary_count == secondary_count else '❌ No'}")
    
    # Revenue comparison
    primary_total = df_sales_primary.agg(sum("TotalAmount")).collect()[0][0]
    secondary_total = df_verify.agg(sum("TotalAmount")).collect()[0][0]
    
    print(f"\n💰 Revenue Comparison:")
    print(f"Primary:   ${primary_total:,.2f}")
    print(f"Secondary: ${secondary_total:,.2f}")
    print(f"Match: {'✅ Yes' if abs(primary_total - secondary_total) < 0.01 else '❌ No'}")
except:
    print("⚠️  Cannot compare - secondary data not available")

## Step 9: Using the Helper Class (Easier Way)

In [ ]:
from scripts.fabric_helper import FabricHelper

# Connect to primary
helper_primary = FabricHelper(use_secondary_db=False)
df_from_primary = helper_primary.read_sql_table(spark, "dbo.Sales")

print("\n" + "="*60)

# Connect to secondary
helper_secondary = FabricHelper(use_secondary_db=True)

# Copy data between servers (simplified)
print("\n📦 Copying data between servers...")
try:
    helper_primary.copy_between_servers(spark, "dbo.Sales", "dbo.SalesCopy")
except Exception as e:
    print(f"Note: {e}")
    print("Make sure FabricDev2 database exists on secondary instance")

## Step 10: Create Aggregated Report on Secondary

In [ ]:
# Create regional summary
df_regional_summary = df_sales_primary.groupBy("Region").agg(
    count("*").alias("TotalOrders"),
    sum("TotalAmount").alias("TotalRevenue"),
    avg("TotalAmount").alias("AvgOrderValue")
).withColumn("ReportGenerated", current_timestamp())

print("✅ Regional summary created:")
df_regional_summary.show()

# Write summary to secondary instance
try:
    df_regional_summary.write.jdbc(
        url=secondary_jdbc_url_dev2,
        table="dbo.RegionalSummary",
        mode="overwrite",
        properties=secondary_properties
    )
    print("\n✅ Regional summary written to SECONDARY instance")
    print("Table: FabricDev2.dbo.RegionalSummary")
except Exception as e:
    print(f"❌ Error: {e}")

## Summary

### ✅ What We Accomplished:

1. Connected to both SQL Server instances
2. Read data from primary instance
3. Transformed data with metadata
4. Wrote data to secondary instance
5. Verified data integrity between instances
6. Created aggregated reports

### 💡 Use Cases:

- **Data Migration Testing** - Test moving data between environments
- **ETL Development** - Source → Transform → Target workflows
- **Data Replication** - Keep multiple instances in sync
- **Backup & Recovery** - Test restore procedures
- **Multi-tenant Development** - Simulate different customer databases

### 🔧 Next Steps:

- Add incremental loading (only new records)
- Implement change data capture (CDC)
- Add data validation checks
- Create automated synchronization jobs
- Add error handling and logging

### 📊 Connections:

- **Primary SQL Server**: localhost:1433 (FabricDev)
- **Secondary SQL Server**: localhost:1434 (FabricDev2)